# Heart Disease Prediction Notebook

In [1]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score


In [2]:
df=pd.read_csv('../datasets/heart/heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df['HeartDisease'].value_counts())

(918, 12)
<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    str    
 2   ChestPainType   918 non-null    str    
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    str    
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    str    
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 97.6 KB
None
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST

In [4]:
df=df.drop_duplicates()

X=df.drop('HeartDisease',axis=1)
y=df['HeartDisease']

In [5]:
categorical_cols=['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope']
X=pd.get_dummies(X,columns=categorical_cols,drop_first=False)
feature_columns=X.columns.tolist()

In [6]:
X_train,X_test,y_train,y_test=train_test_split(
X,y,test_size=0.2,random_state=42,stratify=y)

scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [7]:
models={
'Logistic Regression':LogisticRegression(max_iter=1000),
'Decision Tree':DecisionTreeClassifier(random_state=42),
'Random Forest':RandomForestClassifier(random_state=42),
'KNN':KNeighborsClassifier(),
'SVM':SVC(probability=True),
'Naive Bayes':GaussianNB()
}

results=[]

for name,model in models.items():
    model.fit(X_train_scaled,y_train)
    pred=model.predict(X_test_scaled)
    results.append({
        'Model':name,
        'Accuracy':accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred),
        'Recall':recall_score(y_test,pred),
        'F1 Score':f1_score(y_test,pred)
    })

results_df=pd.DataFrame(results).sort_values(by='Accuracy',ascending=False)
results_df

C:\Users\CodeWithPranav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1 Score
3,KNN,0.918478,0.899083,0.960784,0.928910
2,Random Forest,0.891304,0.894231,0.911765,0.902913
4,SVM,0.885870,0.871560,0.931373,0.900474
0,Logistic Regression,0.885870,0.871560,0.931373,0.900474
5,Naive Bayes,0.885870,0.893204,0.901961,0.897561
1,Decision Tree,0.793478,0.813725,0.813725,0.813725


In [8]:
best_model_name=results_df.iloc[0]['Model']
best_model=models[best_model_name]
print(best_model_name)

MODEL_DIR=Path('../trained_models')
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(best_model,MODEL_DIR/'heart_model.pkl')
joblib.dump(scaler,MODEL_DIR/'heart_scaler.pkl')
joblib.dump(feature_columns,MODEL_DIR/'heart_columns.pkl')
print('Saved Successfully')

KNN
Saved Successfully


In [9]:
model=joblib.load('../trained_models/heart_model.pkl')
scaler=joblib.load('../trained_models/heart_scaler.pkl')
columns=joblib.load('../trained_models/heart_columns.pkl')

sample=pd.DataFrame([{
'Age':55,
'Sex':'M',
'ChestPainType':'ATA',
'RestingBP':130,
'Cholesterol':240,
'FastingBS':0,
'RestingECG':'Normal',
'MaxHR':150,
'ExerciseAngina':'N',
'Oldpeak':1.2,
'ST_Slope':'Up'
}])

sample=pd.get_dummies(sample,columns=['Sex','ChestPainType','RestingECG','ExerciseAngina','ST_Slope'])
sample=sample.reindex(columns=columns,fill_value=0)
sample=scaler.transform(sample)

print(model.predict(sample))
print(model.predict_proba(sample))

[0]
[[0.8 0.2]]
